# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hussaintinwala2/Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My Lane as an ML Task

I am framing **Content Refresh / Opportunity Scoring** as a **ranking task**. The decision is which content pages an editor should review first. Instead of treating every page equally, the system would give pages a priority score and rank them from higher to lower priority. This fits ranking because the main goal is to decide which pages should come first for limited editorial time. This is a provisional framing and may change if the data shows that another task type is more appropriate.


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or Proxy

My provisional proxy is the **change in clicks between the latest 30 days and the previous 30 days**. This is an observed outcome in the starter data, calculated as `clicks_last_30d - clicks_prev_30d`. A negative change indicates that a page received fewer clicks recently, which may make it worth considering for content review. In the 30,000-page starter dataset, 6,806 pages had fewer clicks in the latest 30 days, 6,034 had more clicks, and 17,160 had no change. I am using this as a proxy for potential refresh opportunity, not as proof that a page needs a refresh. I will not use `trend_direction` or `trend_pct` as features because the FlyRank data guide identifies them as derived from the existing trend calculation.


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import files

uploaded = files.upload()

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv


In [20]:
# Calculate observed change in clicks
df["click_change_30d"] = (
    df["clicks_last_30d"] - df["clicks_prev_30d"]
)

print("Pages with fewer clicks:", (df["click_change_30d"] < 0).sum())
print("Pages with more clicks:", (df["click_change_30d"] > 0).sum())
print("Pages with no change:", (df["click_change_30d"] == 0).sum())

Pages with fewer clicks: 6806
Pages with more clicks: 6034
Pages with no change: 17160


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success Metric

My provisional success metric is **Precision@K**. This measures how many of the top K pages in the recommended ranking meet the outcome we define as relevant. This matches the real decision because an editor has limited time and will focus on the highest-priority pages first. For example, if the system recommends 20 pages and 12 meet the chosen outcome, the Precision@20 would be 60%. I will use this as a decision-focused metric rather than judging the system only by overall prediction accuracy. The exact definition of a relevant page and the value of K will need to be tested and refined in later work.


In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check how common our current proxy outcome is
negative_click_change = (df["click_change_30d"] < 0).sum()
total_pages = len(df)

print("Pages with fewer clicks:", negative_click_change)
print("Total pages:", total_pages)
print(
    "Share with fewer clicks:",
    round(negative_click_change / total_pages * 100, 2),
    "%"
)


Pages with fewer clicks: 6806
Total pages: 30000
Share with fewer clicks: 22.69 %


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. The Unit of Analysis, as a Real DataFrame

The unit of analysis is **one pseudonymized content item (page)**. Each row represents one content page and contains information about that page, such as its content characteristics and recent performance. This is the appropriate unit because our decision is to determine which individual pages should be reviewed first. The `content_id` identifies the page, while `client_id` identifies the client it belongs to and is used for grouping rather than as a feature.


In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show a few rows representing individual content pages
sample_cols = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
]

df[sample_cols].head(10)

,content_id,client_id,content_type,impressions_last_30d,clicks_last_30d,sessions_last_30d
0,content_304f48230142,client_f369cb89fc,keyword article,578,2,2
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2501,2,3
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,2382,1,1
3,content_331d6c4de07b,client_19581e27de,keyword article,3626,22,35
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,4211,10,14
5,content_d4084a4bc775,client_f369cb89fc,keyword article,617,0,4
6,content_9a34b442b552,client_8722616204,keyword article,1,0,0
7,content_a63219c6e95a,client_19581e27de,keyword article,636,1,24
8,content_5e6c160719bc,client_6208ef0f77,keyword article,5696,9,36
9,content_c27558df2b0c,client_19581e27de,keyword article,252,0,0


In [23]:
print("Number of content items:", df["content_id"].nunique())
print("Number of rows:", len(df))

Number of content items: 30000
Number of rows: 30000


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML Beats a Fixed Rule Here

A simple rule could prioritize pages based only on one signal, such as giving higher priority to every page whose clicks decreased. However, the starter data contains many signals about each page, including search demand, content characteristics, recent performance, engagement, content age, and search position. The useful pattern may depend on several of these signals together rather than one condition. ML is therefore worth testing because it may combine multiple signals into a more useful ranking than a simple fixed rule. I have not yet shown that ML is better, so this will need to be tested against a simple baseline in later work.


In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the different types of information available for each page
candidate_features = [
    "search_volume",
    "competition",
    "word_count",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "engagement_rate"
]

print("Number of candidate signals:", len(candidate_features))
print("\nCandidate signals:")
for column in candidate_features:
    print("-", column)

Number of candidate signals: 10

Candidate signals:
- search_volume
- competition
- word_count
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- content_age_days
- days_since_last_update
- avg_position
- engagement_rate


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.